# Xây dựng bộ dữ liệu cố định 10.000 ảnh — Fashion200k (Marqo)

**Mục tiêu:** Tạo 3 file `train / val / test` (tỉ lệ 7 / 1 / 2) từ
[`Marqo/fashion200k`](https://huggingface.co/datasets/Marqo/fashion200k) để làm **database chính** cố định
cho toàn bộ thí nghiệm KD của nhóm.

**Cách xử lý trong notebook này:**
1. Gộp các dòng thuộc **cùng 1 sản phẩm** (`product_id` suy ra từ `item_ID`) thành 1 nhóm không thể tách rời.
2. Gộp tiếp các nhóm có **chung caption text** (kể cả khi khác `product_id`) bằng cấu trúc Union-Find,
   vì rủi ro rò rỉ nằm ở *caption trùng lặp*, không chỉ ở *sản phẩm trùng lặp*.
3. Chia 7/1/2 theo **đơn vị nhóm** (không theo đơn vị ảnh) để đảm bảo một nhóm chỉ thuộc đúng 1 split.
4. Kiểm tra lại (assert) sau khi chia: không có `product_id`, không có `caption`, và không có ảnh
   (theo hash nội dung) nào bị trùng giữa 2 split khác nhau.

In [ ]:
# Cài đặt thư viện (Kaggle thường đã có sẵn phần lớn, chỉ cần bổ sung datasets mới nhất)
!pip install -q -U datasets huggingface_hub pillow


In [1]:
import gc
import hashlib
import io
import json
import logging
import os
import random
import shutil
import zipfile
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import pandas as pd
from datasets import load_dataset
from PIL import Image

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("fashion200k_builder")


In [3]:
@dataclass
class DatasetConfig:
    """Cấu hình xây dựng bộ dữ liệu cố định từ Marqo/fashion200k.

    Không hardcode số liệu trong logic phía dưới — mọi tham số đi qua đây.
    """
    hf_dataset_name: str = "Marqo/fashion200k"
    hf_split: str = "data"

    target_total_images: int = 10_000
    train_ratio: float = 0.7
    val_ratio: float = 0.1
    test_ratio: float = 0.2

    random_seed: int = 42
    output_root: str = "/kaggle/working/fashion200k_10k"

    # Dùng cho bước chạy thử (debug) trước khi chạy full — theo đúng thói quen
    # "kiểm tra trên tập nhỏ trước khi chạy toàn bộ" của nhóm.
    debug_target_images: int = 300

    # Ngưỡng cảnh báo: nếu 1 nhóm (product/caption gộp) lớn bất thường,
    # có thể do 1 caption rất chung chung (vd "áo trắng") bị dùng lại cho rất nhiều sản phẩm.
    mega_group_warn_threshold: int = 60

    jpeg_quality: int = 95


cfg = DatasetConfig()

assert abs(cfg.train_ratio + cfg.val_ratio + cfg.test_ratio - 1.0) < 1e-6, (
    f"Tỉ lệ chia phải cộng lại bằng 1.0, hiện tại = "
    f"{cfg.train_ratio + cfg.val_ratio + cfg.test_ratio}"
)
random.seed(cfg.random_seed)
logger.info(f"Config: {cfg}")


17:55:36 | INFO    | Config: DatasetConfig(hf_dataset_name='Marqo/fashion200k', hf_split='data', target_total_images=10000, train_ratio=0.7, val_ratio=0.1, test_ratio=0.2, random_seed=42, output_root='/kaggle/working/fashion200k_10k', debug_target_images=300, mega_group_warn_threshold=60, jpeg_quality=95)


## Bước 1 — Tải metadata (KHÔNG giải mã ảnh) để gom nhóm sản phẩm/caption

`datasets` chỉ giải mã ảnh (`Image` feature → PIL) khi cột `image` thực sự được truy cập.
Ở bước này ta **loại bỏ cột `image`** trước khi chuyển sang `pandas`, nên bước gom nhóm chạy
trên toàn bộ ~202k dòng mà không tốn chi phí decode hàng trăm nghìn ảnh.


In [4]:
def load_metadata_only(cfg: DatasetConfig) -> pd.DataFrame:
    """Tải toàn bộ metadata (không kèm ảnh) của Marqo/fashion200k.

    Args:
        cfg: cấu hình dataset.

    Returns:
        DataFrame với các cột: category1, category2, category3, text, item_ID, row_idx.

    Raises:
        RuntimeError: nếu tải dataset thất bại (mạng, tên dataset sai, v.v.).
    """
    try:
        ds = load_dataset(cfg.hf_dataset_name, split=cfg.hf_split)
    except (OSError, ConnectionError, ValueError) as e:
        logger.error(f"Không tải được dataset '{cfg.hf_dataset_name}': {e}")
        raise RuntimeError("Tải dataset thất bại — kiểm tra kết nối mạng / tên dataset") from e

    logger.info(f"Tổng số dòng trong dataset gốc: {len(ds):,}")

    non_image_cols = [c for c in ds.column_names if c != "image"]
    logger.info(f"Các cột metadata sẽ dùng: {non_image_cols}")

    ds_meta = ds.remove_columns([c for c in ds.column_names if c not in non_image_cols])
    df_meta = ds_meta.to_pandas()
    df_meta["row_idx"] = range(len(df_meta))

    required_cols = {"item_ID", "text"}
    missing = required_cols - set(df_meta.columns)
    if missing:
        raise KeyError(f"Dataset thiếu cột bắt buộc: {missing}")

    return df_meta, ds


df_meta_full, ds_full = load_metadata_only(cfg)
df_meta_full.head()


17:55:45 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Marqo/fashion200k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
17:55:45 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
17:55:46 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Marqo/fashion200k/b9d0e6c9c14337620e442c93f40f7906942fe893/README.md "HTTP/1.1 200 OK"
17:55:46 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Marqo/fashion200k/b9d0e6c9c14337620e442c93f40f7906942fe893/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

17:55:46 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Marqo/fashion200k/resolve/b9d0e6c9c14337620e442c93f40f7906942fe893/fashion200k.py "HTTP/1.1 404 Not Found"
17:55:47 | INFO    | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Marqo/fashion200k/Marqo/fashion200k.py "HTTP/1.1 404 Not Found"
17:55:47 | INFO    | HTTP Request: GET https://huggingface.co/api/datasets/Marqo/fashion200k/revision/b9d0e6c9c14337620e442c93f40f7906942fe893 "HTTP/1.1 200 OK"
17:55:47 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Marqo/fashion200k/resolve/b9d0e6c9c14337620e442c93f40f7906942fe893/.huggingface.yaml "HTTP/1.1 404 Not Found"
17:55:47 | INFO    | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=Marqo/fashion200k "HTTP/1.1 200 OK"
17:55:48 | INFO    | HTTP Request: GET https://huggingface.co/api/datasets/Marqo/fashion200k/tree/b9d0e6c9c14337620e442c93f40f7906942fe893/data?recursive=true&expand=false "HTTP/1.

data/data-00000-of-00009.parquet:   0%|          | 0.00/368M [00:00<?, ?B/s]

17:56:01 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Marqo/fashion200k/resolve/b9d0e6c9c14337620e442c93f40f7906942fe893/data/data-00001-of-00009.parquet "HTTP/1.1 302 Found"


...

17:57:42 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Marqo/fashion200k/resolve/b9d0e6c9c14337620e442c93f40f7906942fe893/data/data-00008-of-00009.parquet "HTTP/1.1 302 Found"


data/data-00008-of-00009.parquet:   0%|          | 0.00/366M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/201624 [00:00<?, ? examples/s]

17:57:57 | INFO    | Tổng số dòng trong dataset gốc: 201,624
17:57:57 | INFO    | Các cột metadata sẽ dùng: ['category1', 'category2', 'category3', 'text', 'item_ID']


,category1,category2,category3,text,item_ID,row_idx
0,dresses,mini and short dresses,green seamed a-line dress,green dress. the dress is made of a shiny mate...,51727804_0,0
1,dresses,casual and day dresses,red contemporary metallic floral print cami dress,dress with a multicolored pattern. the dress h...,54686996_0,1
2,dresses,casual and day dresses,red contemporary metallic floral print cami dress,dress. the dress is multicolored with a patter...,54686996_1,2
3,dresses,casual and day dresses,red contemporary metallic floral print cami dress,dress with a multicolored print. the dress has...,54686996_3,3
4,dresses,casual and day dresses,white eyelet cotton dress,white dress with lace detailing. the dress has...,54946109_0,4


## Bước 2 — Gom nhóm chống rò rỉ (Union-Find: `product_id` + `caption`)

- `product_id` được suy ra từ `item_ID` bằng cách bỏ hậu tố `_<index>` cuối cùng — `[Inferred]`
  dựa trên quan sát dữ liệu mẫu (`54686996_0`, `54686996_1`, `54686996_3` cùng 1 sản phẩm).
  Đây là suy luận theo quy ước đặt tên, **chưa được tài liệu chính thức của Marqo xác nhận**,
  nên cell debug bên dưới sẽ in ra vài ví dụ để tự kiểm tra bằng mắt trước khi chạy full.
- Hai dòng được gộp làm 1 nhóm nếu: cùng `product_id` **HOẶC** cùng `text` (caption) —
  dùng Union-Find để bắc cầu (A trùng caption với B, B trùng product với C → A, B, C cùng 1 nhóm).


In [5]:
class DisjointSet:
    """Cấu trúc Union-Find (path compression) dùng để gộp các khoá (key) thành nhóm liên thông.

    Áp dụng ở đây để đảm bảo mọi dòng cùng product_id HOẶC cùng caption luôn nằm trong
    cùng một nhóm — nhóm này sau đó được gán trọn vẹn vào 1 split duy nhất (train/val/test).
    """

    def __init__(self) -> None:
        self.parent: Dict[str, str] = {}

    def find(self, x: str) -> str:
        self.parent.setdefault(x, x)
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


def extract_product_id(item_id: str) -> str:
    """Suy ra product_id từ item_ID bằng cách bỏ hậu tố '_<chỉ số ảnh>' cuối cùng.

    [Inferred] Quy ước đặt tên item_ID = '<product_id>_<image_index>'.
    """
    item_id = str(item_id)
    return item_id.rsplit("_", 1)[0] if "_" in item_id else item_id


def build_leak_free_groups(df_meta: pd.DataFrame, cfg: DatasetConfig) -> pd.DataFrame:
    """Gán mỗi dòng vào 1 'component' (nhóm chống rò rỉ) dựa trên product_id + caption.

    Args:
        df_meta: DataFrame metadata (phải có cột item_ID, text).
        cfg: cấu hình dataset (dùng cho ngưỡng cảnh báo mega-group).

    Returns:
        DataFrame gốc với 2 cột bổ sung: product_id, component.
    """
    df = df_meta.copy()
    df["product_id"] = df["item_ID"].apply(extract_product_id)

    dsu = DisjointSet()
    for product_id, caption in zip(df["product_id"], df["text"]):
        dsu.union(f"pid::{product_id}", f"cap::{caption}")

    df["component"] = df["product_id"].apply(lambda pid: dsu.find(f"pid::{pid}"))

    comp_sizes = df.groupby("component").size()
    n_mega = int((comp_sizes > cfg.mega_group_warn_threshold).sum())
    if n_mega > 0:
        logger.warning(
            f"Có {n_mega} nhóm với > {cfg.mega_group_warn_threshold} ảnh "
            f"(có thể do caption quá chung chung bị dùng lại nhiều lần). "
            f"Nhóm lớn nhất: {comp_sizes.max()} ảnh."
        )
    logger.info(
        f"Tổng {len(df):,} dòng -> gộp thành {df['component'].nunique():,} nhóm "
        f"(trung vị {comp_sizes.median():.0f} ảnh/nhóm)."
    )
    return df


## Bước 3 — Chia 7/1/2 theo đơn vị NHÓM (không theo đơn vị ảnh)

Thuật toán: xáo trộn thứ tự các nhóm (seed cố định để tái lập), sau đó với mỗi nhóm, gán vào
split đang "thiếu" nhiều nhất so với tỉ lệ mục tiêu (7:1:2), dừng khi tổng số ảnh đã gán đạt
`target_total_images`. Vì gán theo nguyên khối 1 nhóm, tổng số ảnh cuối cùng có thể lệch nhẹ so
với con số mục tiêu — đây là đánh đổi hợp lý để đảm bảo **không rò rỉ dữ liệu**.


In [6]:
def split_groups_no_leak(
    df_grouped: pd.DataFrame, cfg: DatasetConfig, target_total_images: int
) -> Dict[str, List[int]]:
    """Chia các nhóm (component) vào train/val/test theo tỉ lệ mục tiêu, không tách nhóm.

    Args:
        df_grouped: DataFrame đã có cột 'component' (từ build_leak_free_groups).
        cfg: cấu hình (random_seed, train/val/test ratio).
        target_total_images: tổng số ảnh mong muốn (có thể lệch nhẹ do ràng buộc theo nhóm).

    Returns:
        Dict split_name -> danh sách row_idx thuộc split đó.
    """
    components: Dict[str, List[int]] = (
        df_grouped.groupby("component")["row_idx"].apply(list).to_dict()
    )
    component_items = list(components.items())

    rng = random.Random(cfg.random_seed)
    rng.shuffle(component_items)

    target_counts = {
        "train": round(target_total_images * cfg.train_ratio),
        "val": round(target_total_images * cfg.val_ratio),
        "test": round(target_total_images * cfg.test_ratio),
    }
    current_counts = {"train": 0, "val": 0, "test": 0}
    split_assignment: Dict[str, List[int]] = {"train": [], "val": [], "test": []}

    for _component_id, row_indices in component_items:
        if sum(current_counts.values()) >= target_total_images:
            break
        deficits = {
            split: (target_counts[split] - current_counts[split]) / max(target_counts[split], 1)
            for split in target_counts
        }
        best_split = max(deficits, key=deficits.get)
        split_assignment[best_split].extend(row_indices)
        current_counts[best_split] += len(row_indices)

    logger.info(f"Số ảnh thực tế theo split: {current_counts} (mục tiêu: {target_counts})")
    return split_assignment


## Bước 4 — Hàm kiểm tra rò rỉ (product_id / caption / nội dung ảnh)

Chạy độc lập, có thể gọi lại bất kỳ lúc nào để xác nhận 3 split không giao nhau.


In [7]:
def assert_no_leakage(
    df_grouped: pd.DataFrame,
    split_assignment: Dict[str, List[int]],
    image_hashes: Dict[str, str] = None,
) -> None:
    """Kiểm tra không có product_id, caption, hoặc ảnh (theo hash) bị trùng giữa các split.

    Args:
        df_grouped: DataFrame có cột product_id, text, row_idx.
        split_assignment: dict split -> list row_idx.
        image_hashes: (tuỳ chọn) dict row_idx(str) -> md5 hash nội dung ảnh đã giải mã,
            dùng để kiểm tra thêm ở mức ảnh sau khi đã lưu xuống đĩa.

    Raises:
        AssertionError: nếu phát hiện rò rỉ ở bất kỳ tiêu chí nào.
    """
    idx_lookup = df_grouped.set_index("row_idx")

    product_sets: Dict[str, set] = {}
    caption_sets: Dict[str, set] = {}
    for split, row_indices in split_assignment.items():
        sub = idx_lookup.loc[row_indices]
        product_sets[split] = set(sub["product_id"])
        caption_sets[split] = set(sub["text"])

    split_names = list(split_assignment.keys())
    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):
            s1, s2 = split_names[i], split_names[j]

            overlap_pid = product_sets[s1] & product_sets[s2]
            assert not overlap_pid, (
                f"RÒ RỈ product_id giữa {s1} và {s2}: {list(overlap_pid)[:5]} ..."
            )

            overlap_cap = caption_sets[s1] & caption_sets[s2]
            assert not overlap_cap, (
                f"RÒ RỈ caption giữa {s1} và {s2}: {list(overlap_cap)[:3]} ..."
            )

    logger.info("[OK] Không phát hiện rò rỉ product_id hoặc caption giữa các split.")

    if image_hashes:
        hash_to_splits: Dict[str, set] = {}
        for split, row_indices in split_assignment.items():
            for ridx in row_indices:
                h = image_hashes.get(str(ridx))
                if h is None:
                    continue
                hash_to_splits.setdefault(h, set()).add(split)
        leaked_hashes = {h: s for h, s in hash_to_splits.items() if len(s) > 1}
        assert not leaked_hashes, (
            f"RÒ RỈ nội dung ảnh (hash trùng) giữa nhiều split: "
            f"{list(leaked_hashes.items())[:3]} ..."
        )
        logger.info("[OK] Không phát hiện ảnh trùng nội dung giữa các split.")


## Bước 5 — Lưu ảnh + metadata.json cho từng split

Cấu trúc output khớp với hàm `load_split_from_zip()` đã dùng trong notebook huấn luyện KD của nhóm:
mỗi split có `images/<item_ID>.jpg` và `metadata.json` (danh sách dict có khoá `image_file`, `caption`,
`item_ID`, `category1/2/3`).


In [8]:
def save_split_to_disk(
    split_name: str,
    row_indices: List[int],
    df_grouped: pd.DataFrame,
    ds_full,
    cfg: DatasetConfig,
) -> Dict[str, str]:
    """Giải mã và lưu ảnh + metadata.json xuống đĩa cho 1 split.

    Args:
        split_name: 'train' | 'val' | 'test'.
        row_indices: danh sách row_idx (chỉ số trong ds_full) thuộc split này.
        df_grouped: DataFrame metadata đầy đủ (đã có product_id, component).
        ds_full: đối tượng datasets.Dataset gốc (dùng để giải mã ảnh theo row_idx).
        cfg: cấu hình (output_root, jpeg_quality).

    Returns:
        Dict row_idx(str) -> md5 hash nội dung ảnh (dùng cho bước kiểm tra rò rỉ ở mức ảnh).

    Raises:
        RuntimeError: nếu không tạo được thư mục output.
    """
    split_dir = os.path.join(cfg.output_root, split_name)
    images_dir = os.path.join(split_dir, "images")
    try:
        os.makedirs(images_dir, exist_ok=True)
    except OSError as e:
        raise RuntimeError(f"Không tạo được thư mục {images_dir}: {e}") from e

    idx_lookup = df_grouped.set_index("row_idx")
    metadata: List[dict] = []
    image_hashes: Dict[str, str] = {}

    for ridx in row_indices:
        row = idx_lookup.loc[ridx]
        item_id = str(row["item_ID"])
        try:
            pil_img = ds_full[int(ridx)]["image"]
            if pil_img.mode != "RGB":
                pil_img = pil_img.convert("RGB")
        except (KeyError, OSError) as e:
            logger.error(f"Bỏ qua row_idx={ridx} (item_ID={item_id}) do lỗi giải mã ảnh: {e}")
            continue

        image_hashes[str(ridx)] = hashlib.md5(pil_img.tobytes()).hexdigest()

        file_name = f"{item_id}.jpg"
        file_path = os.path.join(images_dir, file_name)
        pil_img.save(file_path, format="JPEG", quality=cfg.jpeg_quality)

        metadata.append({
            "image_file": os.path.join("images", file_name),
            "caption": row["text"],
            "item_ID": item_id,
            "category1": row.get("category1", None),
            "category2": row.get("category2", None),
            "category3": row.get("category3", None),
        })

    meta_path = os.path.join(split_dir, "metadata.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    logger.info(f"[{split_name}] Đã lưu {len(metadata)} ảnh -> {split_dir}")
    return image_hashes


In [9]:
def zip_split(split_name: str, cfg: DatasetConfig) -> str:
    """Nén nội dung BÊN TRONG thư mục split (images/, metadata.json) vào 1 file zip
    ở gốc zip — khớp với cách load_split_from_zip() của notebook huấn luyện giải nén
    (extractall thẳng vào target_dir rồi đọc target_dir/metadata.json).

    Args:
        split_name: 'train' | 'val' | 'test'.
        cfg: cấu hình (output_root).

    Returns:
        Đường dẫn file zip đã tạo.
    """
    split_dir = os.path.join(cfg.output_root, split_name)
    zip_path = os.path.join(cfg.output_root, f"fashion200k_{split_name}.zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _dirs, files in os.walk(split_dir):
            for fname in files:
                abs_path = os.path.join(root, fname)
                rel_path = os.path.relpath(abs_path, start=split_dir)
                zf.write(abs_path, arcname=rel_path)

    logger.info(f"Đã nén {split_dir} -> {zip_path}")
    return zip_path


## CHẠY THỬ / DEBUG — kiểm tra logic trên tập nhỏ trước khi chạy full

Theo đúng thói quen validate từng bước nhỏ trước khi chạy quy mô lớn: chạy toàn bộ pipeline
với `target_total_images` rất nhỏ (`cfg.debug_target_images`) trên một bản sao cấu hình riêng,
in ra vài dòng mẫu để tự kiểm tra bằng mắt (đặc biệt là suy luận `product_id` từ `item_ID`),
rồi mới chạy ở Bước full bên dưới.


In [10]:
from dataclasses import replace

debug_cfg = replace(
    cfg,
    output_root="/kaggle/working/_debug_fashion200k",
)

# Kiểm tra bằng mắt: extract_product_id có hợp lý không?
sample_ids = df_meta_full["item_ID"].astype(str).head(10).tolist()
for sid in sample_ids:
    print(f"item_ID={sid!r:15} -> product_id={extract_product_id(sid)!r}")

df_debug_grouped = build_leak_free_groups(df_meta_full, debug_cfg)
debug_split = split_groups_no_leak(df_debug_grouped, debug_cfg, debug_cfg.debug_target_images)

assert_no_leakage(df_debug_grouped, debug_split)

debug_hashes: Dict[str, str] = {}
for split_name, row_indices in debug_split.items():
    h = save_split_to_disk(split_name, row_indices, df_debug_grouped, ds_full, debug_cfg)
    debug_hashes.update(h)

assert_no_leakage(df_debug_grouped, debug_split, image_hashes=debug_hashes)
print("\n[DEBUG OK] Pipeline chạy đúng trên tập nhỏ. Có thể chạy Bước FULL bên dưới.")

# Dọn dẹp thư mục debug để không lẫn với output chính thức
shutil.rmtree(debug_cfg.output_root, ignore_errors=True)


item_ID='51727804_0'    -> product_id='51727804'
item_ID='54686996_0'    -> product_id='54686996'
item_ID='54686996_1'    -> product_id='54686996'
item_ID='54686996_3'    -> product_id='54686996'
item_ID='54946109_0'    -> product_id='54946109'
item_ID='55570793_0'    -> product_id='55570793'
item_ID='55570793_1'    -> product_id='55570793'
item_ID='55621612_0'    -> product_id='55621612'
item_ID='55621612_1'    -> product_id='55621612'
item_ID='56037632_0'    -> product_id='56037632'


17:58:06 | WARNING | Có 9 nhóm với > 60 ảnh (có thể do caption quá chung chung bị dùng lại nhiều lần). Nhóm lớn nhất: 22269 ảnh.
17:58:06 | INFO    | Tổng 201,624 dòng -> gộp thành 62,884 nhóm (trung vị 2 ảnh/nhóm).
17:58:07 | INFO    | Số ảnh thực tế theo split: {'train': 210, 'val': 30, 'test': 60} (mục tiêu: {'train': 210, 'val': 30, 'test': 60})
17:58:07 | INFO    | [OK] Không phát hiện rò rỉ product_id hoặc caption giữa các split.
17:58:08 | INFO    | [train] Đã lưu 210 ảnh -> /kaggle/working/_debug_fashion200k/train
17:58:08 | INFO    | [val] Đã lưu 30 ảnh -> /kaggle/working/_debug_fashion200k/val
17:58:08 | INFO    | [test] Đã lưu 60 ảnh -> /kaggle/working/_debug_fashion200k/test
17:58:08 | INFO    | [OK] Không phát hiện rò rỉ product_id hoặc caption giữa các split.
17:58:08 | INFO    | [OK] Không phát hiện ảnh trùng nội dung giữa các split.



[DEBUG OK] Pipeline chạy đúng trên tập nhỏ. Có thể chạy Bước FULL bên dưới.


## CHẠY FULL — tạo bộ dữ liệu chính thức 10.000 ảnh (7/1/2)


In [11]:
df_grouped_full = build_leak_free_groups(df_meta_full, cfg)
split_assignment = split_groups_no_leak(df_grouped_full, cfg, cfg.target_total_images)

assert_no_leakage(df_grouped_full, split_assignment)

all_image_hashes: Dict[str, str] = {}
for split_name, row_indices in split_assignment.items():
    hashes = save_split_to_disk(split_name, row_indices, df_grouped_full, ds_full, cfg)
    all_image_hashes.update(hashes)
    gc.collect()

# Kiểm tra rò rỉ lần cuối, lần này có cả kiểm tra nội dung ảnh (md5 hash)
assert_no_leakage(df_grouped_full, split_assignment, image_hashes=all_image_hashes)
print("\n[FULL OK] Đã tạo xong 3 split, không phát hiện rò rỉ dữ liệu.")


17:58:09 | WARNING | Có 9 nhóm với > 60 ảnh (có thể do caption quá chung chung bị dùng lại nhiều lần). Nhóm lớn nhất: 22269 ảnh.
17:58:09 | INFO    | Tổng 201,624 dòng -> gộp thành 62,884 nhóm (trung vị 2 ảnh/nhóm).
17:58:10 | INFO    | Số ảnh thực tế theo split: {'train': 6999, 'val': 1001, 'test': 2003} (mục tiêu: {'train': 7000, 'val': 1000, 'test': 2000})
17:58:10 | INFO    | [OK] Không phát hiện rò rỉ product_id hoặc caption giữa các split.
17:58:26 | INFO    | [train] Đã lưu 6999 ảnh -> /kaggle/working/fashion200k_10k/train
17:58:28 | INFO    | [val] Đã lưu 1001 ảnh -> /kaggle/working/fashion200k_10k/val
17:58:33 | INFO    | [test] Đã lưu 2003 ảnh -> /kaggle/working/fashion200k_10k/test
17:58:33 | INFO    | [OK] Không phát hiện rò rỉ product_id hoặc caption giữa các split.
17:58:33 | INFO    | [OK] Không phát hiện ảnh trùng nội dung giữa các split.



[FULL OK] Đã tạo xong 3 split, không phát hiện rò rỉ dữ liệu.


In [12]:
zip_paths = {split: zip_split(split, cfg) for split in split_assignment}
zip_paths


17:58:42 | INFO    | Đã nén /kaggle/working/fashion200k_10k/train -> /kaggle/working/fashion200k_10k/fashion200k_train.zip
17:58:43 | INFO    | Đã nén /kaggle/working/fashion200k_10k/val -> /kaggle/working/fashion200k_10k/fashion200k_val.zip
17:58:46 | INFO    | Đã nén /kaggle/working/fashion200k_10k/test -> /kaggle/working/fashion200k_10k/fashion200k_test.zip


{'train': '/kaggle/working/fashion200k_10k/fashion200k_train.zip',
 'val': '/kaggle/working/fashion200k_10k/fashion200k_val.zip',
 'test': '/kaggle/working/fashion200k_10k/fashion200k_test.zip'}

## Bước 6 — Thống kê tổng kết (để đối chiếu / báo cáo trong thesis)


In [13]:
idx_lookup_full = df_grouped_full.set_index("row_idx")
summary_rows = []
for split_name, row_indices in split_assignment.items():
    sub = idx_lookup_full.loc[row_indices]
    summary_rows.append({
        "Split": split_name,
        "Số ảnh": len(row_indices),
        "Số sản phẩm (product_id) duy nhất": sub["product_id"].nunique(),
        "Số caption duy nhất": sub["text"].nunique(),
        "Số category1 xuất hiện": sub["category1"].nunique() if "category1" in sub else None,
    })

df_summary = pd.DataFrame(summary_rows)
total_images = df_summary["Số ảnh"].sum()
df_summary["Tỉ lệ thực tế (%)"] = (df_summary["Số ảnh"] / total_images * 100).round(1)
print(f"Tổng số ảnh: {total_images} (mục tiêu: {cfg.target_total_images})")
df_summary


Tổng số ảnh: 10003 (mục tiêu: 10000)


,Split,Số ảnh,Số sản phẩm (product_id) duy nhất,Số caption duy nhất,Số category1 xuất hiện,Tỉ lệ thực tế (%)
0,train,6999,2592,6783,5,70.0
1,val,1001,387,990,5,10.0
2,test,2003,759,1941,5,20.0


In [14]:
# Phân phối category1 theo từng split — để kiểm tra tính đa dạng, không bị lệch hẳn về 1 loại
for split_name, row_indices in split_assignment.items():
    sub = idx_lookup_full.loc[row_indices]
    if "category1" in sub.columns:
        print(f"--- {split_name} ---")
        print((sub["category1"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")
        print()


--- train ---
category1
dresses    30.6%
tops       26.3%
skirts     15.5%
pants      15.0%
jackets    12.6%
Name: proportion, dtype: object

--- val ---
category1
dresses    32.1%
tops       23.1%
skirts     16.9%
jackets    14.7%
pants      13.3%
Name: proportion, dtype: object

--- test ---
category1
dresses    33.9%
tops       21.9%
skirts     16.5%
pants      14.4%
jackets    13.3%
Name: proportion, dtype: object



In [15]:
import os
from huggingface_hub import HfApi, create_repo, login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Lấy token bảo mật từ Kaggle Secrets mà không sợ lộ code
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

username = "qa994"
repo_name = "fashion200k_10k"  # Tên dataset bạn muốn đặt trên HF (có thể đổi tùy ý)
repo_id = f"{username}/{repo_name}"

# ==========================================
# 2. TẠO REPOSITORY TRÊN HUGGING FACE
# ==========================================
try:
    print(f"Đang tạo/kiểm tra repository: {repo_id}...")
    create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    print(f"✅ Đã chuẩn bị xong Repo tại: https://huggingface.co/datasets/{repo_id}")
except Exception as e:
    print(f"⚠️ Có lỗi xảy ra khi tạo repo (hoặc repo đã tồn tại): {e}")

# ==========================================
# 3. ĐỊNH NGHĨA FILE VÀ TIẾN HÀNH UPLOAD
# ==========================================
api = HfApi()

# Đường dẫn gốc tới thư mục chứa các file zip trên Kaggle của bạn
base_dir = "/kaggle/working/fashion200k_10k"

# Danh sách các file cần upload (Khớp chính xác với cấu trúc file trong ảnh của bạn)
files_to_upload = {
    "fashion200k_train.zip": os.path.join(base_dir, "fashion200k_train.zip"),
    "fashion200k_val.zip": os.path.join(base_dir, "fashion200k_val.zip"),
    "fashion200k_test.zip": os.path.join(base_dir, "fashion200k_test.zip")
}

# Vòng lặp upload tự động từng file
for file_in_repo, local_path in files_to_upload.items():
    if os.path.exists(local_path):
        print(f"\n[+] Đang tải lên {file_in_repo} ({os.path.getsize(local_path)/(1024*1024):.2f} MB)...")
        try:
            api.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=file_in_repo,  # Tên file khi xuất hiện trên Hugging Face
                repo_id=repo_id,
                repo_type="dataset"
            )
            print(f"✅ Upload thành công: {file_in_repo}")
        except Exception as e:
            print(f"❌ Lỗi khi tải lên {file_in_repo}: {e}")
    else:
        print(f"⚠️ Không tìm thấy file tại đường dẫn cục bộ: {local_path}")

print("\n🎉 Đã hoàn tất quá trình upload dữ liệu lên Hugging Face!")

18:04:12 | INFO    | HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"


Đang tạo/kiểm tra repository: qa994/fashion200k_10k...


18:04:13 | INFO    | HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 200 OK"


✅ Đã chuẩn bị xong Repo tại: https://huggingface.co/datasets/qa994/fashion200k_10k

[+] Đang tải lên fashion200k_train.zip (208.60 MB)...


18:04:13 | INFO    | HTTP Request: POST https://huggingface.co/api/datasets/qa994/fashion200k_10k/preupload/main "HTTP/1.1 200 OK"
18:04:14 | INFO    | HTTP Request: POST https://huggingface.co/datasets/qa994/fashion200k_10k.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
18:04:14 | INFO    | HTTP Request: GET https://huggingface.co/api/datasets/qa994/fashion200k_10k/xet-write-token/main "HTTP/1.1 200 OK"


...

18:04:37 | INFO    | HTTP Request: POST https://huggingface.co/api/datasets/qa994/fashion200k_10k/preupload/main "HTTP/1.1 200 OK"
18:04:38 | INFO    | HTTP Request: POST https://huggingface.co/datasets/qa994/fashion200k_10k.git/info/lfs/objects/batch "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

18:04:46 | INFO    | HTTP Request: POST https://huggingface.co/api/datasets/qa994/fashion200k_10k/commit/main "HTTP/1.1 200 OK"


✅ Upload thành công: fashion200k_test.zip

🎉 Đã hoàn tất quá trình upload dữ liệu lên Hugging Face!
